In [4]:
!pip install -q -U mlflow boto3 awscli dagshub optuna lightgbm imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 86.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 104.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 92.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [5]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

try:
    
    access_key = user_secrets.get_secret("AWS_ACCESS_KEY")
    secret_key = user_secrets.get_secret("AWS_SECRET_KEY")
    
    # Set environment variables
    os.environ['AWS_ACCESS_KEY_ID'] = access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_key
    os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
    
    print("✅ AWS Credentials successfully loaded!")
    print(f"Access Key ID: {access_key[:4]}****{access_key[-4:]}")
    print(f"Region: {os.environ['AWS_DEFAULT_REGION']}")

except Exception as e:
    print("❌ Failed to find secrets. Make sure you added them in Add-ons -> Secrets.") 

✅ AWS Credentials successfully loaded!
Access Key ID: AKIA****LQZB
Region: us-east-1


In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt

In [7]:
## Setup Mlflow Server
mlflow.set_tracking_uri("http://ec2-54-172-51-46.compute-1.amazonaws.com:5000/")
# Set an experiment
mlflow.set_experiment("LightGbm with HP Tuning")

2026/05/15 15:01:21 INFO mlflow.tracking.fluent: Experiment with name 'LightGbm with HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://amzn-mlflow-bucket-26/6', creation_time=1778857281415, experiment_id='6', last_update_time=1778857281415, lifecycle_stage='active', name='LightGbm with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [8]:
df = pd.read_csv('/kaggle/input/datasets/arnabnath1028/reddit-data/reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [11]:
# --- PREPROCESSING SUMMARY ---
print("[1/5] INITIALIZING: Category remapping & cleaning...")
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})
df = df.dropna(subset=['category'])

ngram_range = (1, 3) 
max_features = 10000 

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'], df['category'], 
    test_size=0.2, random_state=42, stratify=df['category']
)

print(f"📊 [2/5] VECTORIZING: Building {max_features} features...")
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("⚖️ [3/5] BALANCING: Applying SMOTE...")
smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

# --- UPDATED LOGGING FUNCTION ---
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params=None, trial_num=None):
    with mlflow.start_run(nested=True):
        mlflow.set_tag("mlflow.runName", f"{model_name}_Trial_{trial_num}" if trial_num is not None else f"{model_name}_Final")
        if params:
            mlflow.log_params(params)
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        
        mlflow.log_metric("accuracy", acc)
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        return acc

# --- STEP 6: HIGH-ACCURACY LIGHTGBM OBJECTIVE ---
def objective_lightgbm(trial):
    # Optimized search space for 70%+ accuracy
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 10, 20),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'device': 'gpu',        # GPU Acceleration
        'tree_method': 'hist',  # Histogram based
        'verbosity': -1
    }

    model = LGBMClassifier(**params)
    # Log each trial to MLflow
    return log_mlflow("LightGBM_Trial", model, X_train_vec, X_test_vec, y_train, y_test, params, trial.number)

# --- STEP 7: RUNNER WITH AUTO-SAVE & VIZ ---
def run_optuna_experiment():
    print("🏎️ [4/5] SEARCHING: Starting 2-trial GPU Tuning...")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=3)

    print("\n" + "="*50)
    print(f"🏁 WINNER FOUND! Best Accuracy: {study.best_value:.2%}")
    print("="*50)

    # Rebuild the best model
    best_params = study.best_params
    best_model = LGBMClassifier(**best_params, device='gpu', random_state=42)
    
    # Final MLflow log for the winner
    log_mlflow("LightGBM_Final", best_model, X_train_vec, X_test_vec, y_train, y_test, best_params)
    
    # Visualizations
    optuna.visualization.plot_optimization_history(study).show()
    optuna.visualization.plot_param_importances(study).show()
    
    return best_model, vectorizer



[1/5] INITIALIZING: Category remapping & cleaning...
📊 [2/5] VECTORIZING: Building 10000 features...
⚖️ [3/5] BALANCING: Applying SMOTE...


In [12]:
# --- EXECUTION ---
import warnings
import logging

# 1. THE WARNING MUFFLER (Kills the red trash)
warnings.filterwarnings("ignore")
logging.getLogger('mlflow').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.ERROR)
best_model, vectorizer = run_optuna_experiment()

🏎️ [4/5] SEARCHING: Starting 2-trial GPU Tuning...
🏃 View run LightGBM_Trial_Trial_0 at: http://ec2-54-172-51-46.compute-1.amazonaws.com:5000/#/experiments/6/runs/a872b06f3bdf4c7295b3ea754a2918e7
🧪 View experiment at: http://ec2-54-172-51-46.compute-1.amazonaws.com:5000/#/experiments/6
🏃 View run LightGBM_Trial_Trial_1 at: http://ec2-54-172-51-46.compute-1.amazonaws.com:5000/#/experiments/6/runs/2e06b1e68e8d49b685cc789f12670212
🧪 View experiment at: http://ec2-54-172-51-46.compute-1.amazonaws.com:5000/#/experiments/6
🏃 View run LightGBM_Trial_Trial_2 at: http://ec2-54-172-51-46.compute-1.amazonaws.com:5000/#/experiments/6/runs/c7e303f7ae58478aa2fb5720d853b640
🧪 View experiment at: http://ec2-54-172-51-46.compute-1.amazonaws.com:5000/#/experiments/6

🏁 WINNER FOUND! Best Accuracy: 91.52%
🏃 View run LightGBM_Final_Final at: http://ec2-54-172-51-46.compute-1.amazonaws.com:5000/#/experiments/6/runs/b9522b142ce14f54a3ebfda9b62260c2
🧪 View experiment at: http://ec2-54-172-51-46.compute-1.ama

In [14]:
import joblib
print("\n [5/5] EXPORTING: Saving model and vectorizer...")
joblib.dump(best_model, 'best_lightgbm_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')


 [5/5] EXPORTING: Saving model and vectorizer...


['tfidf_vectorizer.pkl']